# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ihSomik/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb)

Simple words, honest numbers. Everything below is **observed** on one sampled month (March 2026) and is **directional**, for decision-support only.

**Lane:** organic search intelligence, the CTR-fix action queue. **Question:** can a model rank pages by *future* missed clicks better than my Week-4 rule, on the same data, the same split and the same metric?

## 1. Method choice and why

### What we predict (label)
My Week-4 rule scored each page by **estimated missed clicks** in the first 21 days of March: `max(expected_ctr - actual_ctr, 0) x impressions`.

For this week, the question is whether that gap **keeps costing clicks**. So for every page that the rule could act on (>=100 impressions, average position 1-20 in the first 21 days):

- **Later missed clicks** = `max(expected_ctr x later_impressions - later_clicks, 0)` over the **last 10 days** of March, with the expected CTR frozen from the first 21 days.
- **Label `y` = 1** when later missed clicks are in the top quarter of the *training* pages (and at least 1 click). So `y = 1` means "still a big, real prize". The threshold is learned from training pages only.

### Inputs (all known by day 21, nothing from the later window)
Impressions, CTR, average position, active days, one-day spike share, position wobble (std of daily position), share of impressions in the last 7 observed days, last-7-day CTR, plus content facts (search volume, word count, char count, competition level) with `_missing` flags. **No client id, no paid, AI, GA4 or product-flag columns, no Week-4 score or expected CTR.**

### Methods from the toolkit, simple first
| Model | Why it is here |
|---|---|
| **Week-4 rule score** | The baseline to beat. No training. |
| **Impressions only** | A sanity floor: if "big pages" alone matches the model, the model adds nothing. |
| **Logistic Regression** | Simple and readable; shows whether a straight-line story is enough. |
| **Random Forest** (shallow) | Catches interactions (e.g. position x impressions) with little tuning. |
| **Gradient Boosting** (small, depth 3) | Safe because it is shallow and validated on unseen clients. |

Complexity is not rewarded: a fancier model only "wins" if it beats the baseline by a clear margin on held-out clients. Otherwise the answer is to keep the rule.

## 2. Split design

- **Time-forward by construction:** features come from days 1-21, the label from days 22-31. The model never sees the future when it scores.
- **Grouped by client (`GroupKFold`):** every client's pages sit entirely in train or entirely in test. This answers the honest question: *will it work for a client it has never seen?* A random row split would let the same client's habits leak across.
- **Same split for every method:** the baseline, the impressions-only floor and the three models are all scored on exactly the same test pages in every fold.
- **Known limits:** the expected-CTR norm is the same pooled Week-4 norm (from all clients' first 21 days), kept identical so the baseline is the real Week-4 rule. Test clients therefore influenced the norm slightly. It is the same for every method, so it does not favour the model.

In [16]:
import os, json
import numpy as np
import pandas as pd
from huggingface_hub import login

# Token comes from Colab Secrets
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN")

if not token:
    raise RuntimeError("HF_TOKEN not found. Add it in Colab -> Secrets and enable notebook access.")

os.environ["HF_TOKEN"] = token
login(token=token)
print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [7]:
!pip install -q huggingface_hub pandas pyarrow scikit-learn

In [8]:
storage_options = {"token": token}
BASE = "hf://datasets/FlyRank/internship-warehouse"

df_raw = pd.read_parquet(f"{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet", storage_options=storage_options)
df_content = pd.read_parquet(f"{BASE}/dim_content.parquet", storage_options=storage_options)
print("Performance rows:", len(df_raw), "| content rows:", len(df_content))

Performance rows: 9841378 | content rows: 519606


In [9]:
# ---- Prep: same windows and same page table as the Week-4 baseline ----
KEY = ["client_hash_id", "content_hash_id"]
need = ["report_date", "client_hash_id", "content_hash_id", "gsc_impressions", "gsc_clicks",
        "gsc_sum_position", "client_has_gsc", "gsc_data_available"]
missing_cols = [c for c in need if c not in df_raw.columns]
assert not missing_cols, f"Missing columns in fact table: {missing_cols} -> rename them in this cell"

df = df_raw[(df_raw["client_has_gsc"] == True) & (df_raw["gsc_data_available"] == True)].copy()
df["report_date"] = pd.to_datetime(df["report_date"])
for c in ["gsc_impressions", "gsc_clicks", "gsc_sum_position"]:
    df[c] = df[c].fillna(0)
df["active"] = (df["gsc_impressions"] > 0).astype(int)

START = df["report_date"].min()
CUTOFF = START + pd.Timedelta(days=20)          # features: first 21 days
obs = df[df["report_date"] <= CUTOFF].copy()
later = df[df["report_date"] > CUTOFF].copy()   # label only, never a feature
print("Feature window:", obs["report_date"].min().date(), "to", obs["report_date"].max().date())
print("Label window:  ", later["report_date"].min().date(), "to", later["report_date"].max().date())

def agg_pages(d):
    return d.groupby(KEY).agg(
        impressions=("gsc_impressions", "sum"), clicks=("gsc_clicks", "sum"),
        sum_pos=("gsc_sum_position", "sum"), days_active=("active", "sum"),
        max_day_impr=("gsc_impressions", "max")).reset_index()

pages = agg_pages(obs)
pages = pages[pages["impressions"] > 0].copy()
pages["ctr"] = pages["clicks"] / pages["impressions"]
raw_pos = pages["sum_pos"] / pages["impressions"]
POSITION_OFFSET = 1 if raw_pos.min() < 1 else 0          # same heuristic as Week 4
pages["avg_position"] = raw_pos + POSITION_OFFSET
pages["spike_share"] = pages["max_day_impr"] / pages["impressions"]

# Week-4 rule, identical settings
MIN_IMPR, MAX_POS, MIN_POS = 100, 20, 1
RULE_BINS = [0, 1.5, 2.5, 3.5, 5, 7, 10, 15, 20]
RULE_LABELS = ["<=1.5", "1.5-2.5", "2.5-3.5", "3.5-5", "5-7", "7-10", "10-15", "15-20"]
pages["eligible"] = (pages["impressions"] >= MIN_IMPR) & (pages["avg_position"] >= MIN_POS) & (pages["avg_position"] <= MAX_POS)
pages["rule_bin"] = pd.cut(pages["avg_position"], bins=RULE_BINS, labels=RULE_LABELS)
norm = (pages[pages["eligible"]].groupby("rule_bin", observed=True)
        .agg(impressions=("impressions", "sum"), clicks=("clicks", "sum")).reset_index())
norm["expected_ctr"] = norm["clicks"] / norm["impressions"]
pages = pages.merge(norm[["rule_bin", "expected_ctr"]], on="rule_bin", how="left")
pages["baseline_score"] = np.where(pages["eligible"],
    (pages["expected_ctr"] * pages["impressions"] - pages["clicks"]).clip(lower=0), 0.0)   # = Week-4 score
print("Eligible pages:", int(pages["eligible"].sum()), "| clients among them:", pages.loc[pages["eligible"], "client_hash_id"].nunique())

Feature window: 2026-03-01 to 2026-03-21
Label window:   2026-03-22 to 2026-03-31
Eligible pages: 68868 | clients among them: 38


In [10]:
# ---- Features (first 21 days + static content facts only) ----
d = obs.copy()
d["pos_day"] = d["gsc_sum_position"] / d["gsc_impressions"].where(d["gsc_impressions"] > 0) + POSITION_OFFSET
pos_std = d.groupby(KEY)["pos_day"].std().rename("pos_std").reset_index()

l7 = d[d["report_date"] > CUTOFF - pd.Timedelta(days=7)]
l7 = l7.groupby(KEY).agg(impr_l7=("gsc_impressions", "sum"), clicks_l7=("gsc_clicks", "sum")).reset_index()

M = pages[pages["eligible"]].copy()
M = M.merge(pos_std, on=KEY, how="left").merge(l7, on=KEY, how="left")
M["pos_std"] = M["pos_std"].fillna(0)
M[["impr_l7", "clicks_l7"]] = M[["impr_l7", "clicks_l7"]].fillna(0)
M["impr_l7_share"] = M["impr_l7"] / M["impressions"]
M["ctr_l7"] = np.where(M["impr_l7"] > 0, M["clicks_l7"] / M["impr_l7"].where(M["impr_l7"] > 0), M["ctr"])

cont = df_content[["content_hash_id", "search_volume", "word_count", "char_count", "competition_level"]].drop_duplicates("content_hash_id")
M = M.merge(cont, on="content_hash_id", how="left")
for c in ["search_volume", "word_count", "char_count"]:
    M[f"{c}_missing"] = M[c].isna().astype(int)
M["comp_missing"] = M["competition_level"].isna().astype(int)
M["competition"] = M["competition_level"].astype(str).str.upper().map({"LOW": 1, "MEDIUM": 2, "HIGH": 3}).fillna(0)
M["log_impr"] = np.log1p(M["impressions"])
M["log_vol"] = np.log1p(M["search_volume"].fillna(0))
M["log_words"] = np.log1p(M["word_count"].fillna(0))
M["log_chars"] = np.log1p(M["char_count"].fillna(0))

FEATURES = ["log_impr", "ctr", "avg_position", "days_active", "spike_share", "pos_std",
            "impr_l7_share", "ctr_l7", "log_vol", "log_words", "log_chars", "competition",
            "search_volume_missing", "word_count_missing", "char_count_missing", "comp_missing"]

# ---- Label source: later missed clicks (expected CTR frozen from the feature window) ----
lp = agg_pages(later)[KEY + ["impressions", "clicks"]].rename(columns={"impressions": "later_impr", "clicks": "later_clicks"})
M = M.merge(lp, on=KEY, how="left")
M[["later_impr", "later_clicks"]] = M[["later_impr", "later_clicks"]].fillna(0)
M["later_missed"] = (M["expected_ctr"] * M["later_impr"] - M["later_clicks"]).clip(lower=0)

M = M.reset_index(drop=True)
M["y"] = np.nan
print("Modeling pages:", len(M), "| clients:", M["client_hash_id"].nunique(), "| features:", len(FEATURES))
print("Share of pages with any later impressions:", round(float((M["later_impr"] > 0).mean()), 3))
print("Later missed clicks - median / 75th pct / max:", round(float(M["later_missed"].median()), 2), round(float(M["later_missed"].quantile(0.75)), 2), round(float(M["later_missed"].max()), 1))
print("\nFeature NaNs (should all be 0):", int(M[FEATURES].isna().sum().sum()))

Modeling pages: 68868 | clients: 38 | features: 16
Share of pages with any later impressions: 0.994
Later missed clicks - median / 75th pct / max: 0.4 1.36 590.9

Feature NaNs (should all be 0): 0


In [11]:
# ---- Leakage guard (inputs only from the feature window and static content) ----
banned = ["paid", "ai_", "ga4_", "later", "label", "product", "flag", "baseline", "expected", "client_hash", "missed"]
bad = [f for f in FEATURES if any(b in f for b in banned)]
assert not bad, f"Banned inputs in FEATURES: {bad}"
assert obs["report_date"].max() <= CUTOFF < later["report_date"].min()
print("Leakage guard PASSED. Features:", FEATURES)

Leakage guard PASSED. Features: ['log_impr', 'ctr', 'avg_position', 'days_active', 'spike_share', 'pos_std', 'impr_l7_share', 'ctr_l7', 'log_vol', 'log_words', 'log_chars', 'competition', 'search_volume_missing', 'word_count_missing', 'char_count_missing', 'comp_missing']


In [12]:
# ---- Grouped, time-forward validation: same folds for every method ----
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

n_clients = M["client_hash_id"].nunique()
n_splits = min(5, n_clients)
assert n_splits >= 3, f"Only {n_clients} clients: grouped validation needs at least 3"
TOP_FRAC = 0.05

def make_models():
    return {
        "Logistic Regression": Pipeline([("sc", StandardScaler()), ("lr", LogisticRegression(max_iter=2000))]),
        "Random Forest (shallow)": RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=20, n_jobs=-1, random_state=42),
        "Gradient Boosting (small)": HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150, random_state=42),
    }

def metrics(y, score, missed):
    # AUC, average precision, precision and captured later-missed-clicks in the top 5% of the fold
    y = np.asarray(y); score = np.asarray(score, float); missed = np.asarray(missed, float)
    k = max(10, int(np.ceil(TOP_FRAC * len(y))))
    top = np.argsort(-score, kind="stable")[:k]
    return {"AUC": roc_auc_score(y, score), "AvgPrecision": average_precision_score(y, score),
            "Precision@top5%": y[top].mean(), "Captured@top5%": missed[top].sum() / max(missed.sum(), 1e-9)}

gkf = GroupKFold(n_splits=n_splits)
rows, fitted = [], {}
for name in ["Gradient Boosting (small)", "Random Forest (shallow)", "Logistic Regression"]:
    M[f"oof_{name}"] = np.nan

for fold, (tr, te) in enumerate(gkf.split(M, groups=M["client_hash_id"])):
    thr = max(float(M.loc[tr, "later_missed"].quantile(0.75)), 1.0)     # threshold from TRAIN pages only
    y_all = (M["later_missed"] >= thr).astype(int)
    ytr, yte = y_all.iloc[tr].to_numpy(), y_all.iloc[te].to_numpy()
    M.loc[M.index[te], "y"] = yte
    M.loc[M.index[te], "fold"] = fold
    if len(set(ytr)) < 2 or len(set(yte)) < 2:
        print(f"Fold {fold}: skipped (one class only)"); continue
    missed_te = M.loc[M.index[te], "later_missed"].to_numpy()

    refs = {"Week-4 rule score (baseline)": M.loc[M.index[te], "baseline_score"].to_numpy(),
            "Impressions only": M.loc[M.index[te], "log_impr"].to_numpy()}
    for nm, sc in refs.items():
        rows.append({"fold": fold, "model": nm, "n_test": len(te), "base_rate": yte.mean(), **metrics(yte, sc, missed_te)})
    for nm, mdl in make_models().items():
        mdl.fit(M.loc[M.index[tr], FEATURES], ytr)
        sc = mdl.predict_proba(M.loc[M.index[te], FEATURES])[:, 1]
        M.loc[M.index[te], f"oof_{nm}"] = sc
        fitted[(nm, fold)] = (mdl, te, yte)
        rows.append({"fold": fold, "model": nm, "n_test": len(te), "base_rate": yte.mean(), **metrics(yte, sc, missed_te)})

res = pd.DataFrame(rows)
print(f"Grouped folds: {n_splits} | clients: {n_clients} | pages: {len(M)}")
print("Pages per test fold:", res.groupby("fold")["n_test"].first().tolist())
print("Label base rate per fold:", res.groupby("fold")["base_rate"].first().round(3).tolist())

Grouped folds: 5 | clients: 38 | pages: 68868
Pages per test fold: [16977, 14325, 12491, 12491, 12584]
Label base rate per fold: [0.345, 0.341, 0.202, 0.208, 0.127]


## 3. Train + compare vs my baseline

*Same data, same metric, same split as the Week-4 baseline. Mean and spread across the held-out client folds.*

In [13]:
cols = ["AUC", "AvgPrecision", "Precision@top5%", "Captured@top5%"]
summary = res.groupby("model")[cols].agg(["mean", "std"]).round(3)
order = ["Week-4 rule score (baseline)", "Impressions only", "Logistic Regression", "Random Forest (shallow)", "Gradient Boosting (small)"]
summary = summary.loc[[m for m in order if m in summary.index]]
print("MODEL vs BASELINE (mean / std over held-out client folds; top 5% = the review queue)")
print(summary.to_string())

# Paired comparison against the baseline, fold by fold (same test pages)
base_f = res[res["model"] == "Week-4 rule score (baseline)"].set_index("fold")
paired = []
for m in order[1:]:
    mf = res[res["model"] == m].set_index("fold")
    common = base_f.index.intersection(mf.index)
    dauc = (mf.loc[common, "AUC"] - base_f.loc[common, "AUC"])
    dcap = (mf.loc[common, "Captured@top5%"] - base_f.loc[common, "Captured@top5%"])
    paired.append({"model": m, "mean_AUC_gain": round(dauc.mean(), 3), "folds_AUC_better": f"{int((dauc > 0).sum())}/{len(common)}",
                   "mean_Captured_gain": round(dcap.mean(), 3), "folds_Captured_better": f"{int((dcap > 0).sum())}/{len(common)}"})
paired = pd.DataFrame(paired)
print("\nPAIRED vs baseline (positive = model better)")
print(paired.to_string(index=False))

# Verdict written by the code from the numbers (not by hand)
ml = paired[paired["model"].isin(["Logistic Regression", "Random Forest (shallow)", "Gradient Boosting (small)"])]
best_row = ml.sort_values("mean_AUC_gain", ascending=False).iloc[0]
BEST = best_row["model"]
gain = best_row["mean_AUC_gain"]
wins = int(best_row["folds_AUC_better"].split("/")[0]); tot = int(best_row["folds_AUC_better"].split("/")[1])
if gain >= 0.02 and wins >= max(1, int(0.8 * tot)):
    VERDICT = "MODEL_BEATS_BASELINE"
elif gain > 0 and wins > tot / 2:
    VERDICT = "SMALL_GAIN_NOT_CLEAR"
else:
    VERDICT = "KEEP_BASELINE_RULE"
print(f"\nBest model by mean AUC gain: {BEST} (gain {gain:+.3f}, better in {wins}/{tot} folds)")
print("VERDICT:", VERDICT)
print({"MODEL_BEATS_BASELINE": "Clear, repeatable gain on unseen clients: worth using the model's ranking.",
       "SMALL_GAIN_NOT_CLEAR": "Gain is small or not repeatable: complexity is not paying for itself yet.",
       "KEEP_BASELINE_RULE": "No gain over the simple rule: keep the rule. That is a valid, useful result."}[VERDICT])
print("Note: BEST is chosen on the same folds it is scored on, so its gain is slightly optimistic.")

MODEL vs BASELINE (mean / std over held-out client folds; top 5% = the review queue)
                                AUC        AvgPrecision        Precision@top5%        Captured@top5%       
                               mean    std         mean    std            mean    std           mean    std
model                                                                                                      
Week-4 rule score (baseline)  0.785  0.016        0.668  0.084           0.904  0.070          0.388  0.026
Impressions only              0.837  0.017        0.552  0.128           0.635  0.137          0.338  0.045
Logistic Regression           0.876  0.007        0.687  0.102           0.830  0.097          0.398  0.042
Random Forest (shallow)       0.891  0.005        0.726  0.087           0.868  0.084          0.338  0.029
Gradient Boosting (small)     0.906  0.003        0.761  0.075           0.914  0.063          0.374  0.033

PAIRED vs baseline (positive = model better)
     

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The cells below show (a) what the best model leans on, using **permutation importance on held-out clients**, (b) what its mistakes look like in its review queue (top 5% per fold), and (c) how much its queue differs from the Week-4 rule's queue.

In [14]:
from sklearn.inspection import permutation_importance

# (a) Permutation importance on held-out folds: how much does AUC drop when one feature is shuffled?
imps = []
for (nm, fold), (mdl, te, yte) in fitted.items():
    if nm != BEST: continue
    Xte = M.loc[M.index[te], FEATURES]
    r = permutation_importance(mdl, Xte, yte, scoring="roc_auc", n_repeats=5, random_state=0, n_jobs=-1)
    imps.append(pd.Series(r.importances_mean, index=FEATURES))
imp = pd.concat(imps, axis=1).mean(axis=1).sort_values(ascending=False)
print(f"PERMUTATION IMPORTANCE for {BEST} (mean AUC drop over held-out folds; ~0 = not used)")
print(imp.round(4).head(10).to_string())

# (b) Review queue = top 5% of each fold by the best model; look at false alarms and misses
M["picked"] = False
for fold in sorted(M["fold"].dropna().unique()):
    idx = M.index[M["fold"] == fold]
    sc = M.loc[idx, f"oof_{BEST}"]
    if sc.isna().all(): continue
    k = max(10, int(np.ceil(TOP_FRAC * len(idx))))
    M.loc[sc.sort_values(ascending=False).index[:k], "picked"] = True
E = M[M["y"].notna()].copy()
E["outcome"] = np.select([E["picked"] & (E["y"] == 1), E["picked"] & (E["y"] == 0), ~E["picked"] & (E["y"] == 1)],
                         ["hit", "false_alarm", "miss"], default="quiet_ok")
print("\nOutcome counts:", E["outcome"].value_counts().to_dict())
print("Queue precision (hits / picked):", round(float((E[E["picked"]]["y"] == 1).mean()), 3), "| base rate:", round(float(E["y"].mean()), 3))
prof = E.groupby("outcome")[["impressions", "avg_position", "ctr", "days_active", "spike_share", "later_missed"]].median().round(2)
print("\nMedian profile by outcome:")
print(prof.to_string())

# Where do picks go wrong by position band?
E["pos_band"] = pd.cut(E["avg_position"], [0, 3, 7, 12, 20], labels=["1-3", "3-7", "7-12", "12-20"])
band = E[E["picked"]].groupby("pos_band", observed=True).agg(n_picked=("y", "size"), precision=("y", "mean")).round(3)
print("\nQueue precision by position band (n printed):")
print(band.to_string())

# (c) Does the model just copy the baseline?
overlap = []
for fold in sorted(E["fold"].unique()):
    f = E[E["fold"] == fold]
    k = max(10, int(np.ceil(TOP_FRAC * len(f))))
    a = set(f.sort_values(f"oof_{BEST}", ascending=False).head(k).index)
    b = set(f.sort_values("baseline_score", ascending=False).head(k).index)
    overlap.append(len(a & b) / k)
print("\nShare of the model's queue that is ALSO in the baseline's queue, per fold:", [round(o, 2) for o in overlap])

PERMUTATION IMPORTANCE for Gradient Boosting (small) (mean AUC drop over held-out folds; ~0 = not used)
log_impr         0.3333
ctr              0.0631
impr_l7_share    0.0238
avg_position     0.0101
ctr_l7           0.0065
log_words        0.0006
days_active      0.0003
pos_std          0.0002
spike_share      0.0001
log_vol          0.0000

Outcome counts: {'quiet_ok': 51127, 'miss': 14295, 'hit': 3171, 'false_alarm': 275}
Queue precision (hits / picked): 0.92 | base rate: 0.254

Median profile by outcome:
             impressions  avg_position  ctr  days_active  spike_share  later_missed
outcome                                                                            
false_alarm       4189.0          7.13  0.0         21.0         0.09          0.26
hit               6552.0          5.57  0.0         21.0         0.09          8.61
miss              1875.0          5.75  0.0         21.0         0.09          2.60
quiet_ok           462.0          7.59  0.0         21.0         0

In [15]:
# ---- Receipt (OK to commit: only counts and metrics, no data rows) ----
os.makedirs("work/outputs", exist_ok=True)
flat = {f"{m}|{c}|{s}": float(summary.loc[m, (c, s)]) for m in summary.index for c in cols for s in ["mean", "std"] if not pd.isna(summary.loc[m, (c, s)])}
metrics_out = {
    "assignment": "ML-08 model vs baseline",
    "feature_window": [str(obs["report_date"].min().date()), str(obs["report_date"].max().date())],
    "label_window": [str(later["report_date"].min().date()), str(later["report_date"].max().date())],
    "split": f"GroupKFold by client, {n_splits} folds, {n_clients} clients",
    "n_pages": int(len(M)), "features": FEATURES,
    "summary_mean_std": flat, "paired_vs_baseline": paired.to_dict(orient="records"),
    "best_model": BEST, "verdict": VERDICT,
    "top_permutation_importance": {k: round(float(v), 4) for k, v in imp.head(5).items()},
    "queue_overlap_with_baseline": [round(float(o), 3) for o in overlap],
}
with open("work/outputs/model_metrics.json", "w") as f:
    json.dump(metrics_out, f, indent=2)
print("Wrote work/outputs/model_metrics.json")

Wrote work/outputs/model_metrics.json


### How to read the results (read the printed numbers first, then write 2-3 sentences of your own)
- **Model vs baseline:** the `VERDICT` line is computed from the paired fold results. A model must beat the rule on *most* folds, not just on average, before it earns a place.
- **The impressions-only row is the honesty check.** If the best model is close to it, most of the "skill" is just "big pages matter".
- **Importance, not cause:** permutation importance shows what the model *uses*, not what *causes* clicks. Features that move together share credit.
- **Error pattern to look for:** false alarms with high spike share or few active days point to unstable pages; misses with low position and low impressions point to small pages that suddenly grew.
- **Overlap with the baseline:** high overlap means the model mostly re-ranks the rule's own picks; low overlap means it found different pages, which needs a manual look before trusting it.

### Honest limits
- One month, one sample, one label definition. Directional, for decision-support only.
- The label is a **proxy** (later CTR gap vs a pooled norm), not a measured result of rewriting a title.
- Page-level data only: branded queries and SERP features can make a low CTR normal.
- The best model is picked on the same folds it is scored on, so its edge is slightly optimistic.
- The pooled CTR norm includes test clients' first 21 days (same for every method).

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled: markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [ ] No client names, URLs, tokens, or private queries anywhere (only hashed ids are used, and never printed in full)
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Model vs baseline table uses the same data, same metric and the same grouped folds
- [ ] I wrote my own 2-3 sentence reading of the error analysis above
- [ ] Committed to my repo under `work/notebooks/` (plus `work/outputs/model_metrics.json`), then submit the repo URL on the card. Done.